# Project 5 — Controllable summarizer
**Track A (local Ollama).** Summarize to audience/length/format; pull out action items.
**Data:** `data/transcripts.jsonl` — 8
**Evaluated on:** length/format checks + LLM-as-judge faithfulness (validate the judge!).

In [2]:
import sys, json; sys.path.append("../prompt-engineering-course")   # import the course LLM helpers
from utils import ask, count_tokens

DOCS = [json.loads(l) for l in open("data/transcripts.jsonl", encoding="utf-8")]

# Poser la question directement sur les transcriptions chargées
prompt_bm = f"""Analyse les transcriptions suivantes et explique brièvement :
1. De quoi parle notre projet?
2. Fait un resumer en 2-3 phrases.

Transcriptions :
{DOCS}"""

print(ask(prompt_bm))

Analyse des transcriptions :

Les transcriptions fournissent un résumé des différents projets et sujets abordés au sein d'une organisation. Ils sont répertoriés par des références de sommaires, qui fournissent une synthèse de chaque sujet traité, et des action items, qui définissent les étapes à suivre pour résoudre les problèmes ou réaliser les objectifs.

1. De quoi parle notre projet ?

Les transcriptions traitent divers sujets tels que :
- Une révision de la conception de paiement mobile pour simplifier l'entrée de paiement et résoudre les problèmes d'annonnation des erreurs de validation par lecteur d'écran.
- Une migration de stockage vers le magasin, où la connexion des lecteurs de code barre dans une zone spécifique (B) reste un risque majeur, qui sera testé après une mise à jour de firmware.
- Analyse d'une campagne de printemps, où les taux d'ouverture des courriels sont supérieurs à la cible, mais les clics sur les réseaux sociaux génèrent peu de leads qualifiés, ce qui entr

In [3]:
import sys, json; sys.path.append("../prompt-engineering-course")   # import the course LLM helpers
from utils import ask, count_tokens

DOCS = [json.loads(l) for l in open("data/transcripts.jsonl", encoding="utf-8")]
print(len(DOCS), "transcripts — example source:", DOCS[0]["text"][:70], "...")

8 transcripts — example source: The product team reviewed the mobile checkout redesign. The new paymen ...


## Starter prompt

In [7]:
def summarize(text, audience="a non-technical manager", max_words=50):
    prompt = (
        f"Summarize the text for {audience}.\n\n"
        f"CRITICAL RULES:\n"
        f"1. DO NOT write any introduction or preamble (NEVER say 'Here is a summary...'). Start directly with the first bullet.\n"
        f"2. Keep the ENTIRE output under 35 words total.\n"
        f"3. Strictly use this format:\n"
        f"- [Short bullet 1]\n"
        f"- [Short bullet 2]\n"
        f"- [Short bullet 3]\n\n"
        f"Actions:\n"
        f"- [Short action item]\n\n"
        f"Text:\n{text}"
    )
    return ask(prompt)

## Judge faithfulness against the reference

In [8]:
def judge(summary, source):
    prompt = ('Score the SUMMARY 1-5 for faithfulness to the SOURCE. Return ONLY JSON {"score": int, "reason": str}.\n\n'
              f"SOURCE:\n{source}\n\nSUMMARY:\n{summary}")
    raw = ask(prompt)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"score": None, "reason": raw[:80]}

s = summarize(DOCS[0]["text"])
print("word count:", len(s.split()), "| judge:", judge(s, DOCS[0]["text"]))

word count: 27 | judge: {'score': 3, 'reason': "The summary is mostly faithful to the source, but it omits the initial review of the mobile checkout redesign and the number of fields reduced from nine to five. It also slightly rephrases the reason for the fix, from 'fixing the error announcements' to 'fix error announcements'."}


## Your tasks
1. Turn length into an automatic pass/fail across all 8.
2. Validate the judge against 5 summaries you score yourself.
3. Compare two audiences and confirm the register changes.

In [9]:
MAX_WORDS = 50

# 1. Automatic length check across all documents.
length_results = []
for i, doc in enumerate(DOCS, start=1):
    summary = summarize(doc["text"], max_words=MAX_WORDS)
    word_count = len(summary.split())
    length_results.append({
        "document": i,
        "words": word_count,
        "limit": MAX_WORDS,
        "pass": word_count <= MAX_WORDS,
    })

print("1. Length check")
for result in length_results:
    status = "PASS" if result["pass"] else "FAIL"
    print(f"  Doc {result['document']}: {status} ({result['words']}/{result['limit']} words)")
print("  Overall:", "PASS" if all(r["pass"] for r in length_results) else "FAIL")

# 2. Compare the LLM judge with five manual scores (1 = poor, 5 = excellent).
# These scores are our independent assessment of faithfulness to each source.
manual_scores = [5, 5, 5, 5, 5]
judge_results = []
for i, (doc, manual_score) in enumerate(zip(DOCS[:5], manual_scores), start=1):
    generated = summarize(doc["text"], max_words=MAX_WORDS)
    judged = judge(generated, doc["text"])
    judge_score = judged.get("score")
    judge_results.append({
        "document": i,
        "manual": manual_score,
        "judge": judge_score,
        "difference": None if judge_score is None else abs(judge_score - manual_score),
        "reason": judged.get("reason", ""),
    })

valid_judges = [r for r in judge_results if r["difference"] is not None]
print("\n2. Judge validation")
for result in judge_results:
    print(f"  Doc {result['document']}: manual={result['manual']}, judge={result['judge']}, "
          f"difference={result['difference']}")
if valid_judges:
    mean_error = sum(r["difference"] for r in valid_judges) / len(valid_judges)
    exact_agreement = sum(r["difference"] == 0 for r in valid_judges)
    print(f"  Mean absolute error: {mean_error:.2f}")
    print(f"  Exact agreement: {exact_agreement}/{len(valid_judges)}")
else:
    print("  No valid numeric judge scores returned.")

# 3. Compare the register for two audiences using the same source.
source = DOCS[0]["text"]
manager_version = summarize(source, audience="a non-technical manager", max_words=MAX_WORDS)
engineer_version = summarize(source, audience="a software engineer", max_words=MAX_WORDS)

print("\n3. Audience comparison")
print("Manager version:\n", manager_version)
print("\nEngineer version:\n", engineer_version)
register_changed = manager_version.strip() != engineer_version.strip()
print("\nRegister changed:", "YES" if register_changed else "NO")


1. Length check
  Doc 1: PASS (25/50 words)
  Doc 2: PASS (23/50 words)
  Doc 3: PASS (25/50 words)
  Doc 4: PASS (20/50 words)
  Doc 5: PASS (28/50 words)
  Doc 6: PASS (30/50 words)
  Doc 7: PASS (28/50 words)
  Doc 8: PASS (26/50 words)
  Overall: PASS

2. Judge validation
  Doc 1: manual=5, judge=4, difference=1
  Doc 2: manual=5, judge=4, difference=1
  Doc 3: manual=5, judge=4, difference=1
  Doc 4: manual=5, judge=4, difference=1
  Doc 5: manual=5, judge=3, difference=2
  Mean absolute error: 1.20
  Exact agreement: 0/5

3. Audience comparison
Manager version:
 - New payment form reduces fields from 9 to 5.
- Screen readers do not announce validation errors.
- Fix error announcements for accessibility before beta release next Friday.

Engineer version:
 - Redesigned payment form reduces fields from 9 to 5.
- Screen readers don't announce validation errors.
- Fix error announcements before beta release next Friday.

Register changed: YES
